In [ ]:
!pip install -q langchain-mistralai langgraph python-dotenv

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage, trim_messages
from langchain_mistralai import ChatMistralAI
from langgraph.graph.message import add_messages

In [ ]:
from google.colab import userdata
import os
os.environ["MISTRAL_API_KEY"] = userdata.get('MISTRAL_API_KEY')

In [ ]:
# DELIBERATELY hardcoded -- see MISTRAL_MODEL_LIMITS.md: mistral-small-latest
# has zero request allowance on many accounts (hard 429).
llm = ChatMistralAI(model="ministral-8b-latest")

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat_node(state: ChatState):

    # trim_messages() shrinks the message list down to fit a token budget
    # BEFORE sending it to the LLM -- this is the core mechanic this whole
    # notebook demonstrates. add_messages (the reducer) keeps EVERY message
    # ever sent in state['messages'] forever; trim_messages() controls what
    # subset of that ever-growing history actually gets shown to the model
    # on THIS call, without deleting anything from state itself.
    trimmed_messages = trim_messages(
        state["messages"],
        max_tokens=100,          # the token BUDGET for what gets kept
        strategy="last",         # keep the MOST RECENT messages that fit
        token_counter=llm,       # use the LLM's own tokenizer to count accurately
        include_system=True,     # always keep a system message if present, regardless of budget
        allow_partial=False,     # never cut a message in half -- drop it whole if it doesn't fit
    )

    response = llm.invoke(trimmed_messages)
    return {"messages": [response]}

In [ ]:
graph_builder = StateGraph(ChatState)

graph_builder.add_node("chat_node", chat_node)

graph_builder.add_edge(START, "chat_node")
graph_builder.add_edge("chat_node", END)

graph = graph_builder.compile()

In [ ]:
messages = [
    HumanMessage(content="Hi, I'm exploring career options after graduation."),
    HumanMessage(content="I have a background in computer science."),
    HumanMessage(content="I'm particularly interested in AI and machine learning."),
    HumanMessage(content="I've worked on a few projects involving NLP."),
    HumanMessage(content="I'm also open to remote roles."),
    HumanMessage(content="Long-term, I want to become a lead ML engineer."),
    HumanMessage(content="What kind of skills should I focus on next?"),
]

In [ ]:
result = graph.invoke({"messages": messages})
print(result["messages"][-1].content)

/usr/local/lib/python3.13/dist-packages/langchain_core/language_models/base.py:463: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

That’s a great career path! To become a **Lead ML Engineer**, you’ll need a mix of **technical depth, leadership skills, and domain expertise**. Since you already have a strong foundation in **AI/ML and NLP**, here’s a structured roadmap to level up:

---

### **1. Strengthen Core ML Engineering Skills**
To transition from a junior ML role to a lead position, you need **production-grade ML expertise**. Focus on:

#### **A. Advanced ML & Deep Learning**
- **Reinforcement Learning (RL)** – Useful for optimization, robotics, and game AI.
  - *Resources*: [RL Course (CS294)](https://web.stanford.edu/class/cs294/), [Stable Baselines3](https://github.com/DLR-RM/stable-baselines3)
- **Generative AI (LLMs, Diffusion Models, GANs)** – High demand in NLP, image generation, and synthetic data.
  - *Resources*: [Hugging Face Course](https://huggingface.co/course/), [Diffusion Models (arXiv)](https://arxiv.org/abs/2006.11239)
- **Neural Architecture Search (NAS)** – Automating model design (useful 

In [ ]:
# Manually run trim_messages the SAME way chat_node does, just to SEE the
# result directly rather than inferring it from the final answer alone.
trimmed = trim_messages(
    messages,
    max_tokens=100,
    strategy="last",
    token_counter=llm,
    include_system=True,
    allow_partial=False,
)

for m in trimmed:
    print(m.content)

Hi, I'm exploring career options after graduation.
I have a background in computer science.
I'm particularly interested in AI and machine learning.
I've worked on a few projects involving NLP.
I'm also open to remote roles.
Long-term, I want to become a lead ML engineer.
What kind of skills should I focus on next?
